# Zarafshan River Basin Water System Optimisation
## A Multi-Objective Optimisation Coursebook

**Authors:** Dr. Tobias Siegfried, Nicolas Lazaro, Dr. Beatrice Marti (hydrosolutions GmbH)  
**License:** CC BY 4.0

---

::: {.callout-note title="Learning Objectives"}
After completing this coursebook, you should be able to:

1.  **Build** physical intuition for water allocation tradeoffs by manually adjusting parameters and observing system response
2.  **Formulate** an optimization problem by enumerating decision variables and selecting which to fix vs. tune
3.  **Explain** Pareto optimality and why no single "best" solution exists when objectives conflict
4.  **Configure** optimization parameters (bounds, constraints, population size, generations)
5.  **Run** NSGA-II and assess convergence using hypervolume diagnostics
6.  **Analyse** Pareto fronts using pairwise scatter plots, parallel coordinates, and representative solution comparison
7.  **Discuss** complexity reduction strategies and their physical justifications
8.  **Interpret** optimized parameter values to understand what the optimizer "learned" about the system
:::

::: {.callout-warning title="Prerequisites"}
This coursebook is the companion to the *Zarafshan River Basin Water System Model: A Simulation-Based Coursebook*. It assumes you have completed that coursebook and are familiar with:

-   The **ZRB system model**: a daily node-edge model with source, storage, splitter, demand, pass-through, sink, and reach components
-   **Water balance** accounting: inflow, consumption, losses, storage change
-   The **SLOP release rule** (Storage Level Operating Policy) for reservoirs
-   **Canal transmission losses**: seepage, evaporation, operational losses
-   The **static-tuned baseline**: one fixed demand-weighted ratio per eligible splitter branch, applied on the daily simulation clock
-   Basic Python proficiency (functions, dictionaries, list comprehensions), pandas, matplotlib/plotly
:::

::: {.callout-note title="Units and Conventions" collapse="true"}
| Symbol / Unit | Meaning |
|:-----------------------------------|:-----------------------------------|
| m³/s | Flow rate (cubic metres per second) |
| m³/day | Daily timestep volume = flow rate × 86,400 s/day |
| Mm³ | Megacubic metres (10⁶ m³) — used for reservoir capacities and annual volumes |
| Bm³ | Billion cubic metres (10⁹ m³) — used for total annual inflow |
| Daily clock | The 2017--2022 simulation runs for 2191 daily timesteps |

Raw flow data are loaded in m³/s and converted to daily timestep volumes internally. Tables report cumulative volumes in Mm³ or Bm³ as appropriate for readability.
:::

# Introduction: From Simulation to Optimisation {#sec-introduction}

## Why Optimise? {#sec-why-optimise}

In the simulation coursebook, different versions of the Zarafshan water system were set up to compare water-distribution heuristics. The simplest heuristic was proportional splitting: each Hydroworks divided incoming flow among its outgoing reaches by fixed proportions, even when this caused spillage or ignored downstream demands.

Next, the static-tuned baseline model represented a system where eligible HydroWorks splitters receive one fixed demand-weighted ratio per downstream branch, applied every day. This heuristic is a meaningful improvement over uniform splitting --- it accounts for the *magnitude* of downstream demand --- but it cannot balance competing objectives.

The static-tuned allocation was computed from total 6-year cumulative demand. It does not adapt to seasonal variation, nor does it consider that allocating more water to agriculture may reduce flows to the Navoi power plant or degrade equity across districts. It answers the question "how should water be distributed on average?" but not "what is the best achievable distribution, given that we care about multiple, and possibly competing, goals simultaneously?"

This coursebook addresses the latter. We shift from **simulation** (evaluating a given policy) to **optimisation** (searching for the best policy). Specifically, we ask: among all feasible parameter settings, which ones minimise agricultural deficits while maintaining power plant reliability, ensuring equitable distribution between users, and keeping operations smooth enough for real-world implementation?

We use the static-tuned model as our starting point for two reasons. First, it keeps allocation logic interpretable: splitter decisions are encoded through policy structures rather than arbitrary free ratios at every branch and timestep (see @sec-decision-space). This is educationally valuable --- students can reason about what each policy controls physically and trace its effect through the system.

Second, the static-tuned model is computationally tractable enough for a teaching workflow while still exposing realistic optimization challenges. Each evaluation simulates 2191 daily timesteps and scores four objectives. The current optimisation problem includes a small number of splitter-policy parameters plus monthly-cyclical reservoir parameters; this lets us discuss dimensionality, convergence, and interpretability without changing the daily runtime clock.

## The Pareto Idea: Why There's No Single "Best" {#sec-pareto-idea}

Suppose you could reduce total agricultural deficit by 100 Mm³, but only at the cost of increasing the power plant's cooling shortfall by 20 Mm³. Is that worth it? What if someone values food security more than industrial reliability --- or vice versa? When objectives conflict, there is no single optimal answer.

Instead, there is a *set* of best-possible compromises: the **Pareto front**. @fig-pareto-idea illustrates the core idea.

On the Pareto front, improving one objective necessarily worsens another. Every point on the front represents a different value judgement about which objectives matter most. The optimizer does not choose for you --- it maps out the entire landscape of achievable tradeoffs, giving decision-makers the information they need to make informed and negotiated choices.

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

In [ ]:
# @title Setup: run this cell first
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
# A runtime that ran this cell earlier may hold an older copy of the course: bring the folder up to date and
# forget any course module Python has already loaded, so that the current code is what gets imported.
if os.path.exists(".git"):
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
for name in [m for m in sys.modules if m == "zarafshan_taqsim" or m.startswith("zarafshan_taqsim.")]:
    del sys.modules[name]
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
# @title Code: fig-pareto-idea
#| label: fig-pareto-idea
#| fig-cap: "Conceptual sketch of a 2-objective tradeoff. Each dot represents a different parameter configuration. Blue dots on the Pareto front are *non-dominated*: no other configuration is better on both objectives simultaneously. Grey dots are *dominated*: at least one front solution beats them on every objective. The shaded region is unachievable."
from zarafshan_taqsim import figures, plot_pareto_idea, tables

figures.reset()
tables.reset()
fig = plot_pareto_idea()
figures.caption(
    fig,
    "Conceptual sketch of a 2-objective tradeoff. Blue dots on the Pareto front are non-dominated; grey dots are dominated; the shaded region is unachievable.",
    label="pareto-idea",
)

 We will develop this formally in @sec-pareto-formal after building hands-on intuition in @sec-building-intuition.

## Planning for the Future {#sec-planning}

The Zarafshan basin faces converging pressures. Samarkand's population continues to grow, driving irrigation and municipal demand upward. Glacial retreat in the headwaters reduces the summer flow pulse that the reservoirs depend on. Ageing canal infrastructure degrades conveyance efficiency. These trends do not operate in isolation --- they compound: a growing population demands more water from a shrinking supply delivered through deteriorating infrastructure.

The optimisation framework developed in this coursebook is not limited to the historical 2017--2022 sequence used here. Because TaqSim [@taqsim] is a general-purpose event-sourced simulation engine, it can ingest any hydrological input sequence --- synthetic droughts, projected glacier retreat scenarios, or demand growth trajectories --- and produce the corresponding system behaviour. TaqSim's optimisation module then searches for Pareto-optimal allocations under those modified conditions. The entire pipeline (define scenarios → simulate → optimise → analyse Pareto front) can be re-run with different input data without modifying any code.

This capability opens the door to **scenario-based planning**. A water manager can ask: "If summer inflow decreases by 30% due to glacier retreat, how does the Pareto front shift? Which allocation strategies remain viable, and which collapse?" By running the optimizer under multiple plausible futures --- a drought sequence, a high-growth demand scenario, a canal rehabilitation programme --- and comparing the resulting Pareto fronts, decision-makers can identify **robust allocations** that perform reasonably well across a range of futures, rather than policies optimised for a single historical sequence that may never recur.

This is a key aspect of long-term basin planning: the Pareto front is not a fixed answer but a map that shifts as conditions change. The tools used in this coursebook --- TaqSim for simulation and multi-objective optimisation --- are designed to make this re-mapping fast and systematic, enabling adaptive management as the basin's hydrological and socio-economic context evolves.

## Coursebook Roadmap {#sec-roadmap}

The ten sections of this coursebook follow a progressive arc:

1.  **Introduction** (this section): Motivation and the Pareto idea.
2.  **Setup and Baseline Recap** (@sec-baseline): Establish the computational environment and the "before" benchmark.
3.  **Building Intuition** (@sec-building-intuition): Manually adjust parameters to discover tradeoffs firsthand.
4.  **The Decision Space** (@sec-decision-space): Enumerate the tunable splitter and reservoir policy parameters.
5.  **The Objective Space** (@sec-objective-space): Present 9 candidate objectives, select 4, and evaluate the baseline.
6.  **Optimisation Concepts** (@sec-optimisation-concepts): Pareto theory, NSGA-II, and the optimizer framework.
7.  **Optimisation Results** (@sec-optimisation-results): Run NSGA-II and analyse the Pareto front.
8.  **Discussion** (@sec-discussion): What optimisation reveals and what it cannot.
9.  **Conclusions** (@sec-conclusions): Key findings and practical implications.
10. **Exercise Collection** (@sec-exercises): All exercises with cross-references.

::: {.callout-important title="Key Takeaways -- Introduction"}
-   The static-tuned baseline is a good heuristic but cannot balance competing objectives.
-   When objectives conflict, there is no single optimal solution --- there is a Pareto front of best-possible compromises.
-   Optimisation maps out the tradeoff landscape; *human values* select the solution.
:::

# Setup and Baseline Recap {#sec-baseline}

## Setup {#sec-setup}

In [ ]:
# @title Code: setup
#| label: setup
#| echo: true
import logging

import matplotlib.pyplot as plt

from zarafshan_taqsim import (
    N_TIMESTEPS,
    apply_static_demand_splitting,
    baseline_district_summary,
    baseline_objective_table,
    build_objectives,
    compute_baseline,
    compute_reference_point,
    create_zrb_system,
    decision_table_from_schema,
    figures,
    tables,
    front_arrays,
    kattakurgan_policy_variants,
    narpay_equity_variants,
    objective_catalog_table,
    parameter_schema_summary,
    plot_convergence,
    plot_narpay_equity,
    plot_parallel_coordinates,
    plot_pareto_concept,
    plot_pareto_3d,
    plot_pareto_scatter,
    plot_release_sensitivity,
    plot_representative_district_deficits,
    plot_representative_sink_flows,
    plot_selected_parameter_profiles,
    plot_tunable_parameter_map,
    release_comparison_table,
    representative_indices,
    representative_table,
    run_coursebook_optimization,
    simulate_representatives,
    visualize_network,
)

# Constants
TIMESTEPS = N_TIMESTEPS
SEED = 42

# Plotting defaults
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

logging.basicConfig(level=logging.WARNING)

## The Static-Tuned Baseline {#sec-static-baseline}

The static-tuned baseline system was developed in the simulation coursebook. It uses `create_zrb_system(use_canal_losses=True)` --- including realistic canal transmission losses --- combined with `apply_static_demand_splitting()`, which computes one fixed demand-weighted ratio for each eligible splitter branch from cumulative downstream demand. The system still runs on the daily clock; the split ratios are simply repeated each day.

In [ ]:
# @title Code: baseline-setup
#| label: baseline-setup
baseline = create_zrb_system(use_canal_losses=True)
apply_static_demand_splitting(baseline, verbose=False)
baseline.validate()
baseline.simulate(TIMESTEPS)

@fig-network-map shows the geographic layout of the Zarafshan water system. Water enters at the Source node in the upper reaches, flows through a network of HydroWorks splitters (orange) that distribute it among irrigation districts (red), reservoirs (green), and sinks (grey). The two reservoirs --- Kattakurgan and Akdarya --- provide storage and regulated releases. The Powerplant (purple) receives cooling water from HW_Karmana. Understanding this spatial layout is essential for interpreting the optimisation results: upstream allocation decisions propagate through the network and affect every downstream node.

In [ ]:
# @title Code: fig-network-map
#| label: fig-network-map
#| fig-cap: "Geographic layout of the Zarafshan water system. Node positions correspond to UTM Zone 42N coordinates. Colours indicate node type: blue = Source, green = Storage (reservoirs), orange = Splitter (HydroWorks), red = Demand (irrigation districts), purple = PassThrough (power plant), grey = Sink."
fig, _ = visualize_network(baseline, show_reaches=False, figsize=(18, 12))
figures.caption(
    fig,
    "Geographic layout of the Zarafshan water system. Node positions correspond to UTM Zone 42N coordinates and colours indicate node type.",
    label="network-map",
)

In [ ]:
# @title Code: tbl-baseline-districts
#| label: tbl-baseline-districts
#| tbl-cap: "Per-district satisfaction under the static-tuned baseline. Deficit is the cumulative unmet demand over 2191 daily timesteps."
tables.caption(
    baseline_district_summary(baseline, TIMESTEPS).style.hide(axis="index"),
    "Per-district satisfaction under the static-tuned baseline. Deficit is the cumulative unmet demand over 2191 daily timesteps.",
    label="baseline-districts",
)

The static-tuned baseline serves as the benchmark throughout this coursebook. The optimizer must demonstrate that it can do better --- and at what cost to other objectives. @tbl-baseline-districts shows per-district demand satisfaction as a key performance metric for the static-tuned baseline system. For more information about the system setup, consult the water system simulation chapter of the coursebook.

::: {.callout-important title="Key Takeaways -- Baseline"}
-   The static-tuned baseline uses one fixed demand-weighted ratio per eligible splitter branch on the daily clock.
-   Canal losses are enabled, providing a realistic benchmark.
-   Some districts (particularly those furthest downstream) show larger deficits.
:::

# Building Intuition: Try Before You Optimise {#sec-building-intuition}

Before we let the optimizer explore a large number of different parameter combinations, we will manually adjust parameters and observe the consequences. This builds the physical intuition that makes the Pareto front meaningful rather than abstract. Students who have struggled with tradeoffs manually will understand the optimizer's output as a quantification of their own experience.

## What Happens If We Change the Reservoir Policy? {#sec-release-sensitivity}

The static-tuned baseline uses a SLOP (Storage Level Operating Policy) for Kattakurgan reservoir with four parameters: the target release rate `vr`, two storage zone thresholds `v1` (buffer zone top) and `v2` (conservation zone top), and a `buffer_coef` that controls how aggressively the reservoir releases when storage is in the buffer zone between dead storage and `v1`. What happens if we adopt a fundamentally different operating philosophy?

We compare three reservoir policies that represent distinct operational strategies. The **conservative** policy sets a low release rate, a high conservation zone threshold, and a minimal buffer coefficient --- it hoards water aggressively, releasing only when storage is high. The **baseline** uses the static-tuned defaults. The **aggressive** policy lowers the conservation zone, raises the buffer coefficient, and maintains a high release rate --- it pushes water downstream as fast as possible.

In [ ]:
# @title Code: fig-release-sensitivity
#| label: fig-release-sensitivity
#| fig-cap: "Effect of Kattakurgan reservoir operating policy on storage (left) and downstream flow at HW_Karmana (right). The conservative policy hoards water and sustains storage, but restricts downstream flow during peak season. The aggressive policy drains the reservoir rapidly, boosting peak-season downstream flow but leaving little for the autumn recession."
variants = kattakurgan_policy_variants(TIMESTEPS)
fig = plot_release_sensitivity(variants, TIMESTEPS)
figures.caption(
    fig,
    "Effect of Kattakurgan reservoir operating policy on storage and downstream flow at HW_Karmana under conservative, baseline, and aggressive release settings.",
    label="release-sensitivity",
)

The aggressive policy drains the reservoir rapidly, boosting downstream flow during peak irrigation season but leaving little storage for the September--October recession when downstream districts depend on reservoir releases to bridge the gap. The conservative policy hoards water aggressively --- sustaining high storage levels throughout the simulation --- but restricts downstream flow precisely when irrigation demand is highest. The baseline sits between these extremes. **There is no single "right" reservoir policy** --- each represents a different bet on when water is most valuable.

In [ ]:
# @title Code: tbl-release-comparison
#| label: tbl-release-comparison
#| tbl-cap: "Effect of Kattakurgan reservoir operating policy on system objectives. The three policies produce substantially different deficit and spillage outcomes, illustrating the tradeoff between water conservation and downstream supply."
tables.caption(
    release_comparison_table(variants).style.hide(axis="index"),
    "Effect of Kattakurgan reservoir operating policy on system objectives. The three policies produce substantially different deficit and spillage outcomes, illustrating the tradeoff between water conservation and downstream supply.",
    label="release-comparison",
)

## Can You Improve Equity Manually? {#sec-equity-manual}

The baseline shows that Karmanakonimex --- the district furthest downstream --- typically has the largest deficit. Can we fix this by sending more water downstream?

HW_Narpay splits flow between two reach branches: one toward HW_Confluence (which feeds HW_Karmana and ultimately Karmanakonimex) and one toward Narpay district. The baseline static-tuned ratio is demand-derived. Let us test two alternative allocations: one that diverts more water to Narpay, and one that sends more water downstream toward Confluence.

In [ ]:
# @title Code: fig-equity-manual
#| label: fig-equity-manual
#| fig-cap: "Per-district deficit under three HW_Narpay allocation ratios. Sending more water toward Confluence supports downstream Karmanakonimex; diverting more to Narpay reduces local deficit but leaves less water for the downstream corridor."
equity_variants = narpay_equity_variants(baseline, TIMESTEPS)
fig = plot_narpay_equity(equity_variants)
figures.caption(
    fig,
    "Per-district deficit under three HW_Narpay allocation ratios, illustrating the local-versus-downstream equity tradeoff.",
    label="equity-manual",
)

The results illustrate the core tradeoff. Diverting more water to Narpay reduces local deficit but can increase Karmanakonimex's deficit because less water reaches HW_Karmana downstream. Conversely, sending more water downstream helps Karmanakonimex but leaves Narpay with less. The baseline ratio represents one particular compromise, but it is not unique --- the sequential network topology means that **helping one district inevitably hurts another**. This is the fundamental equity challenge that multi-objective optimisation addresses: rather than choosing one compromise, we map out all achievable tradeoffs.

## The Impossible Tradeoff {#sec-impossible-tradeoff}

Can you find a *single* parameter change that simultaneously reduces total deficit (O1), maintains zero power plant shortfall (O4), AND improves equity (O6)?

The answer is: no. Each attempt improves one objective at the expense of another. **Multi-objective conflicts are structural**, not solvable by cleverness. This is exactly the problem Pareto optimisation addresses --- it maps out *all* best-possible compromises simultaneously.

In @sec-optimisation-results, we will let the optimizer explore thousands of parameter combinations and reveal the full tradeoff surface. But first, we need to define precisely *what* we can control (@sec-decision-space), *what* we care about (@sec-objective-space), and *how* the optimizer searches (@sec-optimisation-concepts).

::: {.callout-tip title="Exercise 5: Manual Tradeoff Discovery" collapse="true"}
**Task:** Choose two parameters from the static-tuned baseline (e.g., Kattakurgan `vr` and the HW_Narpay ratio). Create at least 4 different configurations by varying these parameters. For each, simulate and record O1 (total agricultural deficit) and O6 (equity / max district deficit). Plot your 4+ attempts as points in O1-vs-O6 space.

1.  Can you find a configuration that beats the baseline on *both* objectives simultaneously?
2.  Among your attempts, which ones are "non-dominated" --- i.e., no other attempt is better on both objectives?
3.  After completing @sec-pareto-front, overlay the optimizer's Pareto front on your manual attempts. How close did you get?

**Hint:** Use `copy.deepcopy(system)` to create independent copies for each variant. Modify the release rule via `object.__setattr__()` as shown in the baseline setup.
:::

::: {.callout-important title="Key Takeaways -- Building Intuition"}
-   Releasing more water from Kattakurgan helps downstream districts but risks draining the reservoir before the autumn recession.
-   Improving equity at one node worsens the deficit at another --- the sequential network topology creates structural conflicts.
-   No single parameter change can simultaneously improve all objectives. Multi-objective optimisation is the right tool for mapping these tradeoffs.
:::

# The Decision Space: What Can We Control? {#sec-decision-space}

## The Full Knob Inventory {#sec-full-knobs}

The current optimisation problem exposes **124 tunable scalar values**. Four of them are splitter `eflow_fraction` values; the remaining 120 are monthly-cyclical SLOP parameters for the two reservoirs (`vr`, `v1`, `v2`, `buffer_coef`, and `flood_coef`, each with 12 calendar-month values per reservoir). The simulation still runs daily; the monthly parameter tuples are a compact seasonal operating policy repeated across the daily clock.

In [ ]:
# @title Code: tbl-full-knob-inventory
#| label: tbl-full-knob-inventory
#| tbl-cap: "Current optimization parameter inventory discovered from WaterSystem.param_schema(). Reservoir parameters are monthly-cyclical values; splitter e-flow fractions are scalar values."
schema_summary = parameter_schema_summary(baseline)
tables.caption(
    schema_summary.style.hide(axis="index"),
    "Current optimization parameter inventory discovered from WaterSystem.param_schema(). Reservoir parameters are monthly-cyclical values; splitter e-flow fractions are scalar values.",
    label="full-knob-inventory",
)

The parameter inventory is discovered directly from the system rather than hand-maintained in the notebook. **EFlowSplitPolicy** splitters expose a single tunable `eflow_fraction` that controls the nature-vs-irrigation split; the remainder is distributed by demand-weighted ratios (computed from historical district demands, not tuned). **MonthlyDistribution** and **PriorityDistribution** splitters have fixed allocations. Each reservoir contributes five SLOP parameter families with 12 seasonal entries each. An `Ordered(v1 ≤ v2)` constraint ensures zone boundaries remain properly nested for each reservoir policy.

@fig-knob-map-full shows the same geographic network from @fig-network-map, with Reach nodes collapsed into directed edges, but now highlights the six tuneable components. Each tuneable node is annotated with the number of scalar values it contributes to the optimizer.

In [ ]:
# @title Code: fig-knob-map-full
#| label: fig-knob-map-full
#| fig-cap: "Geographic network map highlighting the 6 tuneable components and their scalar parameter counts, with Reach nodes collapsed into directed edges. Splitters contribute one e-flow fraction each; reservoirs contribute monthly-cyclical SLOP values."
fig = plot_tunable_parameter_map(baseline, schema_summary)
figures.caption(
    fig,
    "Geographic network map highlighting the tuneable components and their scalar parameter counts, with Reach nodes collapsed into directed edges.",
    label="knob-map-full",
)

## Design Choices: From Knobs to Parameters {#sec-reducing-complexity}

The decision space is a deliberate compromise between realism and interpretability. We do not optimize arbitrary daily split ratios everywhere in the network. Instead, we expose a small set of policy-level splitter variables and seasonal reservoir rule parameters.

**EFlowSplitPolicy (4 corridor splitters, 1 param each = 4 params).** Each corridor splitter uses an `EFlowSplitPolicy` that separates water into two streams: a fraction `eflow_fraction` goes to natural/environmental reaches, and the remainder `(1 - eflow_fraction)` is distributed among irrigation reaches by demand-weighted ratios. The demand weighting is computed from historical district demand data, not tuned by the optimizer. This design collapses what would be N−1 free ratio parameters per splitter into a single scalar --- a dramatic reduction for HW_Ravatkhoza (which has 4 targets) and HW_Damkhoza (4 targets). The physical justification: the nature-vs-irrigation tradeoff is the meaningful operational decision; how the irrigation share is divided among districts is better governed by demand patterns than by unconstrained optimization.

**PriorityDistribution at HW_Dargom (0 params).** The regulatory minimum flow to Sink_Kashkadarya must be served first, with the remainder flowing to Dargom district. This is a hard operational constraint, not a tunable parameter.

**Fixed `MonthlyDistribution` policies at HW_Narpay and HW_Confluence (0 params each).** In the current daily model, these policies hold a 365-day ratio cycle. For the static-tuned baseline, the same demand-weighted ratio is repeated every day. HW_Narpay distributes to 2 targets with fixed ratios. HW_Confluence has a single downstream target (HW_Karmana), so there is no allocation choice.

**Monthly-cyclical SLOP at both reservoirs (60 values each = 120 values).** Both RES_Kattakurgan and RES_Akdarya use `ZRBReleaseRule` with five seasonal parameter families: `vr`, `v1`, `v2`, `buffer_coef`, and `flood_coef`. Each family has 12 calendar-month entries that repeat over the daily simulation. An `Ordered(v1 ≤ v2)` constraint ensures the buffer zone top remains below the conservation zone top.

::: {.callout-note title="Why Not Optimize Canal Losses?" collapse="true"}
The \~578 canal loss parameters (seepage coefficients, canal dimensions, lining types) represent fixed infrastructure. Changing them would mean physically rehabilitating canals --- a capital investment decision, not an operational decision. The optimizer controls operational policy: how to allocate and release water given the current infrastructure.
:::

In [ ]:
# @title Code: tbl-reduced-config
#| label: tbl-reduced-config
#| tbl-cap: "Policy-level decision variables and declared bounds."
tables.caption(
    decision_table_from_schema(schema_summary).style.hide(axis="index"),
    "Policy-level decision variables and declared bounds.",
    label="reduced-config",
)

The bounds in @tbl-reduced-config reflect the declared `__bounds__` on each strategy class. **E-flow fractions** span [0, 1]: 0 means no reserved environmental corridor share, 1 means all water is reserved for environmental targets up to the configured cap. **SLOP `vr`** ranges from 0 to 500 m³/s. **`v1` and `v2`** range from 0 to 10⁹ m³, with `Ordered(v1 ≤ v2)` enforced by taqsim's built-in repair function. **`buffer_coef`** spans [0, 1], while **`flood_coef`** spans [0, 5]. The optimizer discovers values within these bounds; the simulation then enforces physical storage, dead-storage, capacity, and loss constraints.

The important distinction is between the **daily simulation clock** and the **monthly-cyclical operating policy**. The optimizer does not set 2191 daily reservoir values. It sets 12 seasonal values per SLOP parameter family, and the release rule applies the appropriate value at each daily timestep.

::: {.callout-tip title="Exercise D: Counting Decision Variables" collapse="true"}
**Task:** The current system has 4 splitter e-flow fractions and 120 reservoir SLOP values.

1.  Verify the 120 reservoir values from the five SLOP parameter families and 12 calendar-month entries per family.
2.  Using the rule of thumb `pop_size ≥ 10 × variables`, what population size would a full 124-variable optimization suggest?
3.  What are the practical consequences for computation time and Pareto-front coverage?
4.  How could seasonal grouping reduce dimensionality while preserving the daily simulation clock?
:::

::: {.callout-tip title="Exercise E: Priority Rules and Optimizer Signals" collapse="true"}
**Task:** HW_Karmana currently exposes an `eflow_fraction` policy parameter rather than arbitrary branch ratios.

1.  Explain how changing `eflow_fraction` at HW_Karmana can affect Powerplant cooling reliability even though the powerplant objective is evaluated downstream.
2.  Canal losses on the HW_Karmana → Powerplant edge can reduce the flow that actually reaches the Powerplant. Does this mean O4 is a pure upstream signal, or does local infrastructure still play a role? Explain.
3.  If we wanted the optimizer to control all HW_Karmana branch ratios directly, what new tradeoff would this enable? Would the added dimensionality be worth it?
:::

::: {.callout-important title="Key Takeaways -- Decision Space"}
-   The current optimisation schema has 124 scalar values: 4 splitter e-flow fractions and 120 monthly-cyclical reservoir SLOP values.
-   EFlowSplitPolicy collapses N-1 splitter ratios into a single e-flow fraction with demand-weighted remainder distribution.
-   Ordered(v1 ≤ v2) constraints on both reservoirs are enforced by taqsim's built-in repair.
-   The exposed variables control policy-level allocation and release behaviour without optimizing a separate value for every daily timestep.
:::

# The Objective Space: What Do We Care About? {#sec-objective-space}

## The Objective Landscape {#sec-objective-landscape}

Nine candidate objectives are discussed for the Zarafshan basin, each capturing a different dimension of system performance. The current executable optimisation uses the four objectives exposed by `build_objectives()`: O1, O4, O6, and O9.

In [ ]:
# @title Code: tbl-objective-landscape
#| label: tbl-objective-landscape
#| tbl-cap: "All nine candidate objectives for the Zarafshan basin. The four selected for optimisation are highlighted."
tables.caption(
    objective_catalog_table().style.hide(axis="index"),
    "All nine candidate objectives for the Zarafshan basin. The four selected for optimisation are highlighted.",
    label="objective-landscape",
)

Each objective captures a distinct dimension of how the water system performs. Understanding what each one measures --- and why it matters --- is essential before deciding which to optimise.

**O1 — Total agricultural deficit** sums the cumulative unmet irrigation demand across all six districts over the full 2191-day simulation. It is computed by the `total_agricultural_deficit()` function, which aggregates `DeficitRecorded` events at each demand node. This is the primary performance metric: irrigation accounts for over 90% of water use in the Zarafshan basin, and reducing total deficit directly benefits food production and rural livelihoods. We select O1 for optimisation.

**O2 — Downstream environmental flow deficit** would measure how much the flow at Sink_Navoi falls below the regulatory minimum required for downstream environmental and municipal uses. It captures the tension between upstream irrigation extraction and downstream commitments. It is not part of the current executable four-objective helper because it is largely determined by upstream allocation decisions already captured by O1 and O4; including it would add dimensionality without independent tradeoff information.

**O3 — Reservoir spillage** would sum the total uncontrolled spillage from both reservoirs. Spillage represents wasted water: inflow that the reservoir cannot store or release through controlled outlets. It is correlated with O1 (good allocation implies less spillage) and is not selected as an independent objective.

**O4 — Power plant cooling deficit** measures the cumulative shortfall at the Powerplant node relative to its minimum required cooling flow of 15 m³/s. Computed by `passthrough_min_flow_deficit()`, it tracks how often and by how much the Powerplant receives less water than it needs. O4 reflects both upstream allocation choices and the local HW_Karmana policy that determines how much water remains available for the powerplant branch after environmental and irrigation allocations. We select O4 for optimisation.

**O5 — Inter-basin transfer deficits** would measure the shortfall at the three sink nodes (Sink_Jizzakh, Sink_Kashkadarya, Sink_Navoi), which represent commitments to adjacent basins and regulatory minimum flows. Like O2, these flows are largely residuals of upstream allocation decisions and do not add independent tradeoff dimensions. Not selected.

**O6 — Equity (maximum district deficit)** takes the maximum cumulative deficit across all six districts. While O1 measures the total, O6 measures the worst-off district. A system with low total deficit but one district bearing most of the shortfall would score well on O1 but poorly on O6. Computed by `equity_deficit()`, this objective captures the fairness dimension that aggregate metrics miss. We select O6 for optimisation.

**O7 — Temporal deficit concentration** would measure how deficits are distributed across time: are shortfalls spread evenly over the year, or concentrated in a few critical months? A system that delivers water reliably most of the year but fails catastrophically during peak season scores worse than one with moderate but consistent deficits. This objective is not yet implemented.

**O8 — Storage sustainability** would measure whether reservoir storage levels at the end of the simulation are comparable to initial levels. A policy that achieves low deficits by draining reservoirs to zero is not sustainable over multi-year horizons. This objective is not yet implemented.

**O9 — Release variability** measures the cumulative day-to-day change in Kattakurgan's release volumes over the daily simulation. Computed by `release_variability()`, it penalises policies with erratic release patterns. Real reservoir operators cannot implement policies that swing wildly between timesteps --- operational smoothness is a practical constraint. We select O9 for optimisation.

Notice that all nine objectives are formulated as **minimisation** problems: total deficit, spillage, variability, maximum deficit --- in every case, less is better. This is a convention, not a limitation. Any maximisation objective can be converted to minimisation by negating it: minimising $-O$ is mathematically equivalent to maximising $O$. We adopt the minimisation convention because NSGA-II and most multi-objective optimisation frameworks assume minimisation by default. Formulating all objectives as "minimise" keeps the implementation clean and avoids the need for sign-flipping logic in the evaluation function. When interpreting results, remember that **lower values are always better** on every objective axis.

## Selecting Our Four Objectives {#sec-four-objectives}

We select **O1, O4, O6, O9** for the following reasons:

-   **O1 (Total agricultural deficit)** is the primary system purpose --- irrigation accounts for \>90% of water use.
-   **O4 (Power plant cooling deficit)** is a quasi-hard constraint. It measures whether enough water is routed through the downstream network and through HW_Karmana's local allocation policy to satisfy the powerplant cooling requirement. Canal losses on the HW_Karmana → Powerplant edge can also contribute, so it is not a pure upstream-only signal.
-   **O6 (Equity)** captures the fairness dimension that O1 alone misses. Total deficit can be low even if one district is sacrificed.
-   **O9 (Release variability)** ensures solutions are operationally implementable. A release policy with wild day-to-day swings cannot be executed by real operators.

::: {.callout-note title="Why Not Include All Nine Objectives?" collapse="true"}
More objectives means a higher-dimensional Pareto front that is harder to cover with a finite population and harder to visualise. O2 and O5 are largely determined by what remains after irrigation and storage; including them adds dimensionality without independent tradeoff information. O3 (spillage) is correlated with O1 (good allocation implies less spillage). O7 and O8 are not yet implemented.
:::

In [ ]:
# @title Code: objectives-setup
#| label: objectives-setup
#| echo: true
OBJECTIVES = build_objectives(timesteps=TIMESTEPS)
OBJ_NAMES = [obj.name for obj in OBJECTIVES]
baseline_scores = compute_baseline(baseline, OBJECTIVES, timesteps=TIMESTEPS)
reference_point = compute_reference_point(baseline_scores)

## Baseline Objective Values {#sec-baseline-objectives}

In [ ]:
# @title Code: tbl-baseline-objectives
#| label: tbl-baseline-objectives
#| tbl-cap: "Baseline objective values for the static-tuned system. These are the benchmarks the optimizer must beat."
tables.caption(
    baseline_objective_table(baseline_scores, OBJ_NAMES).style.hide(axis="index"),
    "Baseline objective values for the static-tuned system. These are the benchmarks the optimizer must beat.",
    label="baseline-objectives",
)

The baseline provides a concrete benchmark. The optimizer's task is to find parameter configurations that improve on these values --- knowing that improving one objective typically worsens another.

::: {.callout-tip title="Exercise F: Adding a Fifth Objective" collapse="true"}
**Task:** Suppose you added O2 (downstream environmental flow deficit at Sink_Navoi) as a 5th optimisation objective.

1.  The current 4-objective Pareto front is a 3D hypersurface. What dimensionality would a 5-objective front have? Using the population sizing guidelines from @sec-optimizer-config, what minimum `pop_size` would you recommend?
2.  O2 would measure flow deficit at Sink_Navoi. The power plant (O4) also depends on flow reaching the downstream network. Would O2 and O4 be positively correlated, negatively correlated, or independent? What would strong correlation mean for the shape of the front?
3.  If O2 and O5 were implemented with the same sink minimum-flow deficit pattern, what would distinguish them? Could you combine them into a single objective without losing information?
:::

::: {.callout-important title="Key Takeaways -- Objective Space"}
-   Nine candidate objectives capture different aspects of system performance.
-   We select four (O1, O4, O6, O9) that span the key tradeoff dimensions: quantity, reliability, equity, and operability.
-   The baseline provides the benchmark that the optimizer must improve upon.
:::

# Optimisation Concepts {#sec-optimisation-concepts}

## The Pareto Front --- Formal Treatment {#sec-pareto-formal}

Building on the intuitive introduction in @sec-pareto-idea, we now define Pareto optimality formally.

**Dominance.** Solution $\mathbf{a}$ *dominates* solution $\mathbf{b}$ (written $\mathbf{a} \prec \mathbf{b}$) if and only if:

-   $\mathbf{a}$ is no worse than $\mathbf{b}$ on every objective, **and**
-   $\mathbf{a}$ is strictly better than $\mathbf{b}$ on at least one objective.

**Pareto front.** The Pareto front (or non-dominated set) is the set of all solutions that are not dominated by any other feasible solution. Non-dominated sorting assigns a *rank* to each solution: rank 0 = the Pareto front, rank 1 = the front that emerges after removing rank 0, and so on.

**Hypervolume.** The hypervolume indicator measures the "volume" of objective space dominated by the Pareto front, relative to a reference point. A larger hypervolume means a better-converged, better-spread front. We use it to track convergence over generations. For visualization, the hypervolume is shown in a stylized manner in @fig-pareto-concept.

The **reference point** must be chosen carefully. A natural first thought is to use the baseline's objective values, but this does not work. The whole point of multi-objective optimisation is that Pareto-front solutions *trade off* objectives: a solution that excels on O1 (low agricultural deficit) may deliberately sacrifice O6 (equity) or O9 (smoothness) to achieve that gain. Such a solution can be *worse* than the baseline on some individual objectives while being much better on others --- and it is still genuinely non-dominated (no other solution beats it on all four objectives simultaneously). If the reference point equals the baseline value on each axis, any Pareto solution that exceeds the baseline on even one objective would fall outside the hypervolume box on that axis, and its contribution to the indicator would be clipped or lost entirely. The hypervolume would undercount the front's true quality.

We therefore use **1.5× the baseline's objective values** as per-objective reference points. The multiplier provides headroom so the entire Pareto front is captured, including solutions that deliberately sacrifice one objective to excel on others. The reference point represents a "known bad" ceiling --- performance so poor that any reasonable allocation should beat it on every axis --- not a performance target. The four objectives have very different scales: O1, O4, and O6 are cumulative deficit volumes, while O9 is cumulative day-to-day release variation. The 1.5× multiplier is applied per-objective to ensure each axis has proportionate headroom.

**Knee points** represent solutions where the tradeoff slope changes most sharply --- the "maximum bang for buck" compromises. Moving away from a knee in any direction trades large degradation on one objective for small improvement on another.

In [ ]:
# @title Code: fig-pareto-concept
#| label: fig-pareto-concept
#| fig-cap: "Formal 2D Pareto front illustration showing dominance, the non-dominated front (rank 0), a knee point, and the hypervolume indicator (shaded area)."
fig = plot_pareto_concept()
figures.caption(
    fig,
    "Formal 2D Pareto front illustration showing dominance, the non-dominated front, a knee point, and the hypervolume indicator.",
    label="pareto-concept",
)

## NSGA-II {#sec-nsga2}

Searching for Pareto-optimal solutions requires an algorithm that can explore a high-dimensional decision space and simultaneously optimise four conflicting objectives. We use a **genetic algorithm** --- an optimisation method inspired by biological evolution. The idea is intuitive: start with a *population* of candidate solutions (each one a different parameter vector), evaluate how well each performs on all four objectives, then preferentially *select* the better-performing candidates, *combine* them (crossover) to produce offspring that inherit traits from both parents, and *mutate* some values slightly to explore new regions of the search space. After many generations of this cycle, the population converges toward the Pareto front.

In the Zarafshan context, each "individual" in the population is a vector of e-flow fractions and monthly-cyclical reservoir SLOP values. Crossover blends two parent vectors to produce a child that might combine the reservoir policy of one parent with the e-flow fractions of another. Mutation introduces small perturbations --- nudging a release rate up by 2 m³/s, or shifting a zone threshold by 10 Mm³ --- to prevent the population from getting stuck in a local optimum.

For multi-objective problems, the challenge is that there is no single "best" solution to converge toward. **NSGA-II** (Non-dominated Sorting Genetic Algorithm II) [@deb2002nsga2] addresses this by replacing simple fitness ranking with two mechanisms. First, **non-dominated sorting** ranks all individuals into Pareto layers: rank 0 consists of the non-dominated solutions (the current best Pareto front), rank 1 consists of solutions that would be non-dominated if rank 0 were removed, and so on. When selecting parents for the next generation, lower-rank solutions are preferred. Second, **crowding distance** measures how isolated each solution is within its rank --- solutions in sparsely populated regions of the objective space are preferred over those in crowded clusters. This dual selection pressure drives the population simultaneously toward the true Pareto front (convergence) and across it (diversity), producing a well-spread approximation of the entire tradeoff surface. NSGA-II has become the standard algorithm for multi-objective water resources optimisation [@maier2014evolutionary; @reed2013evolutionary].

We implement this using TaqSim's `taqsim.optimization.optimize()` path via the package's `run_optimization()` wrapper. It uses SBX crossover, polynomial mutation, and the auto-generated repair function. The optimizer accepts the objectives, number of timesteps, population size, generation count, random seed, and number of parallel workers, managing the full population dynamics, selection, generational tracking, and Pareto-front recording.

## The Optimisation Question {#sec-opt-question}

::: {.callout-important title="The Optimisation Question"}
*Given policy-level e-flow fractions and monthly-cyclical reservoir SLOP values, what combination minimises agricultural deficits (O1) while maintaining power plant reliability (O4), ensuring equitable distribution across districts (O6), and keeping operations smooth enough for real-world implementation (O9)?*

*There is no single answer. The Pareto front maps out every best-possible compromise.*
:::

::: {.callout-tip title="Exercise G: Effect of Freeing More Knobs" collapse="true"}
**Task:** The current optimisation uses monthly-cyclical reservoir SLOP values. Suppose we replaced the 12-month cycles with four seasonal values per SLOP family.

1.  How many reservoir SLOP values would remain?
2.  Would the Pareto front likely move closer to the origin, further away, or stay the same? Why?
3.  What practical advantage might the four-season formulation have, even if the 12-month formulation produces better objective scores?
:::

::: {.callout-important title="Key Takeaways -- Optimisation Concepts"}
-   Pareto optimality means no solution on the front can be improved on one objective without worsening another.
-   Hypervolume measures front quality: convergence and spread.
-   NSGA-II uses non-dominated sorting and crowding distance to maintain diverse Pareto approximations.
-   The repair function enforces the `Ordered(v1 ≤ v2)` constraint after every genetic operation.
:::

# Optimisation Results {#sec-optimisation-results}

## Optimizer Configuration {#sec-optimizer-config}

Each candidate solution is a vector of policy parameters. The optimizer proposes a vector, the system is configured with those values, simulated for 2191 daily timesteps, and scored on 4 objectives. The best solutions are combined and mutated to produce the next generation.

::: {.callout-note title="Under the Hood: How taqsim Discovers Parameters" collapse="true"}
The `WaterSystem.param_schema()` method auto-discovers all tunable parameters by walking the node tree and collecting fields declared in `__params__`. For our system, this yields the same schema summarised in @tbl-full-knob-inventory: splitter `eflow_fraction` values plus monthly-cyclical reservoir SLOP values.

The `WaterSystem.constraint_specs()` method discovers inter-parameter constraints (here, 2 `Ordered(v1 ≤ v2)` constraints). The optimizer's `make_repair()` factory builds a repair function that enforces these constraints after every genetic operation: if `v1 > v2`, they are swapped.

The `with_vector()` method creates an immutable clone of the system with a new parameter vector, avoiding the manual `deepcopy` + `object.__setattr__()` patching pattern.
:::

In [ ]:
# @title Code: optimizer-problem-setup
#| label: optimizer-problem-setup
#| echo: true
#| eval: false
system = create_zrb_system(use_canal_losses=True)
apply_static_demand_splitting(system, verbose=False)
system.validate()

# Inspect what the optimizer will tune
schema = system.param_schema()
print(f"Tunable parameters: {len(schema)}")
for spec in schema:
    print(f"  {spec.path} = {spec.value:.4f}")

# Inspect constraints
constraints = system.constraint_specs()
print(f"\nConstraints: {len(constraints)}")
for spec in constraints:
    print(f"  {type(spec.constraint).__name__}({spec.constraint.params}) @ {spec.prefix}")

In [ ]:
# @title Code: run-optimization
#| label: run-optimization
#| echo: true
#| eval: false
FORCE_RERUN = False
SMOKE_TEST = True

result, convergence_history, config = run_coursebook_optimization(
    system,
    OBJECTIVES,
    OBJ_NAMES,
    reference_point,
    timesteps=TIMESTEPS,
    seed=SEED,
    smoke_test=SMOKE_TEST,
    force_rerun=FORCE_RERUN,
)
print(f"Pareto front: {len(result.solutions)} non-dominated solutions")

In [ ]:
# @title Code: extract-front
#| label: extract-front
#| echo: true
#| eval: false
# Extract objective arrays from the result.
front_obj, n_front = front_arrays(result, OBJ_NAMES)
print(f"Pareto front: {n_front} non-dominated solutions")

The population size for the genetic algorithm must be large enough to maintain diversity across the Pareto front. With 4 objectives, the front is a 3D hypersurface --- far harder to cover than a 1D curve. The following rule of thumb is applied when choosing population size.

| Objectives | Front geometry  | Typical pop_size |
|:-----------|:----------------|:-----------------|
| 2          | 1D curve        | 50--100          |
| 3          | 2D surface      | 100--200         |
| 4          | 3D hypersurface | 150--300         |

A second rule of thumb requires the population size to be at least 10 times the number of decision variables. With the current schema, this would imply a much larger run than is practical for a live teaching notebook. The code therefore defaults to `SMOKE_TEST = True`, using a tiny `12 × 2` run only to verify that the pipeline executes. For a real teaching/demo run, set `SMOKE_TEST = False`, which uses `POP_SIZE = 300` and `N_GENERATIONS = 100`. For publication-quality fronts, the run should be repeated with a larger population and longer generation count.

The number of generations determines how long the algorithm has to refine and converge the Pareto front. In smoke mode, the computational budget is only 24 evaluations. In full demo mode, 300 individuals per generation times 100 generations yields 30,000 total evaluations.

## Convergence Diagnostics {#sec-convergence}

Before examining the Pareto front, we need to verify that the algorithm has converged --- that is, that additional generations would not meaningfully improve the front. The primary convergence diagnostic is the **hypervolume indicator**: the volume of objective space dominated by the current Pareto front, measured relative to the reference point defined in @sec-pareto-formal. As the algorithm progresses, the front improves and the hypervolume grows. When the hypervolume plateaus --- stops increasing from one generation to the next --- the front has stabilised and further computation yields diminishing returns.

@fig-convergence shows two diagnostics. The left panel tracks hypervolume over generations: a steep rise in early generations (the algorithm rapidly discovers good solutions) followed by a plateau (the front has converged). If this curve were still rising at generation 100, we would increase `N_GENERATIONS` to 200 or 300 and re-run. The right panel shows the number of non-dominated solutions (front size) over generations. This typically grows quickly as the algorithm populates the front, then stabilises as new solutions merely replace existing ones rather than expanding the front.

In [ ]:
# @title Code: fig-convergence
#| label: fig-convergence
#| eval: false
#| fig-cap: "Hypervolume convergence over generations. A plateauing curve indicates the Pareto front has stabilised."
if convergence_history.records:
    fig = plot_convergence(convergence_history)
    figures.caption(
        fig,
        "Hypervolume convergence over generations. A plateauing curve indicates the Pareto front has stabilised.",
        label="convergence",
    )
else:
    print("No convergence history available; cached results may have been loaded without callback records.")

If the hypervolume curve has not plateaued, `N_GENERATIONS` should be increased (e.g., to 200 or 300) and set `FORCE_RERUN = True`. Each additional generation adds `POP_SIZE` evaluations, so runtime scales almost linearly with both population size and generation count.

## The Pareto Front {#sec-pareto-front}

@fig-pareto shows six scatter plots --- one for each pair of the four objectives. Every blue dot is a solution on the Pareto front: no other solution in the population is better on *all four* objectives simultaneously. The red star is the static-tuned baseline for reference.

In [ ]:
# @title Code: fig-pareto
#| label: fig-pareto
#| eval: false
#| fig-cap: "Pairwise scatter plots of the 4-objective Pareto front. Each blue dot is a non-dominated solution; the red star marks the static-tuned baseline. The 6 panels show all C(4,2) = 6 objective pairs."
fig = plot_pareto_scatter(result, OBJ_NAMES, baseline_scores=baseline_scores)
figures.caption(
    fig,
    "Pairwise scatter plots of the 4-objective Pareto front. Each blue dot is a non-dominated solution; the red star marks the static-tuned baseline.",
    label="pareto",
)

### Reading the Scatter Matrix {#sec-reading-scatter-matrix}

A natural first reaction is: "Why not connect the dots to draw the Pareto front as a line?" In a two-objective problem, this is exactly the right thing to do. The Pareto front is a 1-dimensional curve, and sorting solutions by one objective automatically orders them along the front. You can connect them left-to-right and get the familiar concave trade-off curve shown in textbooks.

With four objectives, the situation is fundamentally different. The Pareto front lives in a **4-dimensional objective space** and forms a **3-dimensional hypersurface** (one dimension fewer than the number of objectives). Each scatter panel projects this 3D surface onto a 2D plane by discarding two of the four objectives. This projection has three important consequences:

**1. Interior points are not inferior.** A solution that appears to sit "inside" the point cloud in the O1 vs. O4 panel is not dominated --- it is on the true 4D Pareto front. It appears interior only because it excels on O6 or O9, the two objectives *not shown* in that panel. If we drew a 2D convex envelope connecting only the boundary points of O1 vs. O4, we would visually dismiss solutions that are genuinely Pareto-optimal. They would look like failures, when in fact they represent deliberate compromises on objectives that the panel cannot display.

**2. Dominance is not transitive across panels.** A solution on the boundary of the O1 vs. O4 panel may sit deep inside the cloud in the O6 vs. O9 panel, and vice versa. No single 2D envelope captures the full trade-off structure. To understand *why* a solution is on the Pareto front, you must look at all six panels together (or use the parallel coordinates plot in @sec-parallel-coordinates, which shows all objectives simultaneously).

**3. The point cloud's shape reveals the trade-off structure.** Even without connecting dots, the scatter panels carry rich information:

- **Tight negative correlation** (points along a diagonal from upper-left to lower-right): the two objectives are in strong direct conflict. Improving one reliably worsens the other. Look for this in O1 vs. O4 --- reducing agricultural deficit increases power plant cooling shortfall.
- **Wide cloud with no clear trend**: the two objectives are relatively independent, or their relationship is mediated by the hidden objectives. The optimizer can find solutions across a wide range of both objectives.
- **Triangular or L-shaped clouds**: one objective hits a hard physical limit (a floor or ceiling) while the other varies freely. This reveals structural constraints in the system.
- **Baseline position**: Where the red star sits relative to the cloud shows how much the optimizer improved over the heuristic allocation. If the baseline is inside or near the edge of the cloud, the static-tuned heuristic was already close to optimal for that pair. If it is far from the cloud, the optimizer found substantial gains.

::: {.callout-note title="Why 6 Panels?" collapse="true"}
With $k$ objectives there are $\binom{k}{2}$ unique pairs, so 4 objectives produce $\binom{4}{2} = 6$ panels. This is the minimum number of 2D views needed to show every pairwise relationship. However, even all 6 panels together cannot fully represent the 3D Pareto hypersurface --- just as six photographs of a sculpture do not capture every angle. The parallel coordinates plot (@fig-parallel-coordinates) complements the scatter matrix by showing all four objectives on a single axis system, making it easier to trace how individual solutions balance all objectives simultaneously.
:::

Exercise 6 (@sec-pareto-front) asks you to apply these reading strategies to the scatter matrix above.

## Exploring the 4D Pareto Front in 3D {#sec-3d-pareto}

The scatter matrix projects the 4D Pareto front onto six 2D planes. A complementary view is to show three objectives simultaneously in a rotatable 3D scatter plot, using the fourth objective as a colour dimension.

@fig-pareto-3d plots O1 (agricultural deficit) on the x-axis, O4 (power plant deficit) on the y-axis, and O6 (equity) on the z-axis. Each point is coloured by its O9 (smoothness) value. This effectively uses colour as a fourth visual dimension.

Rotate the 3D view by clicking and dragging. As you do, notice that points that appeared "interior" in the 2D scatter panels actually lie on the surface of the 3D point cloud --- they are genuinely non-dominated, just projected inward by the missing dimensions. Look for whether the colour gradient clusters in one region of the front: demanding smoother operations (lower O9) can restrict the achievable tradeoffs among the other three objectives.

In [ ]:
# @title Code: fig-pareto-3d
#| label: fig-pareto-3d
#| eval: false
#| fig-cap: "Interactive 3D Pareto front: O1 × O4 × O6 with O9 as colour. Drag to rotate the front."
fig = plot_pareto_3d(result, OBJ_NAMES)
figures.caption(
    fig,
    "Interactive 3D Pareto front: O1 by O4 by O6 with O9 as colour. Drag to rotate the front.",
    label="pareto-3d",
)

## Interactive Parallel Coordinates {#sec-parallel-coordinates}

The parallel coordinates plot is the single most powerful tool for navigating a multi-objective Pareto front. In @fig-parallel-coordinates, each vertical axis represents one quantity --- either an objective value or a key decision variable. Each polyline connecting values across the axes represents one Pareto-front solution. The colour encodes O1 (total agricultural deficit), so dark-coloured lines correspond to solutions with low total deficit and light-coloured lines to solutions with high deficit.

**How to interact with it.** Click and drag along any vertical axis to select a range. This filters the display to show only solutions whose value on that axis falls within the selected range. Multiple axis filters can be combined: drag on O4 to select only solutions with zero power plant deficit, then drag on O6 to further restrict to equitable solutions. The remaining lines show exactly which O1 and O9 values --- and which parameter settings --- are achievable under those combined constraints. To clear a filter, click on the axis outside the selected range.

**What can be learned.** Lines that run roughly parallel between two axes (without crossing) indicate objectives that are positively correlated: improving one tends to improve the other. Lines that cross heavily between two axes indicate conflicting objectives: the front spans a wide range of tradeoffs between them. On the decision variable axes, parameters where all lines cluster tightly at a similar value are "locked in" by the physics of the system --- the optimizer agrees on their setting regardless of which objective is prioritised. Parameters where lines spread widely across the axis range are the true operational levers: different tradeoff strategies require fundamentally different settings.

**As a stakeholder negotiation tool**, the parallel coordinates plot makes the "price" of combining constraints transparent and quantifiable. Consider this workflow: (1) a power company representative filters O4 to select only solutions with zero cooling deficit; (2) an agriculture ministry official further filters O1 below a tolerable threshold; (3) the equity commission constrains O6 to ensure no district bears a disproportionate share of the shortfall. What remains after these three filters reveals the achievable range of O9 (smoothness) and the specific parameter settings (splitter ratios, reservoir policies) that these solutions require. The parallel coordinates plot turns abstract Pareto analysis into a concrete negotiation framework: stakeholders can see, in real time, what they gain and what they pay for each additional constraint.

In [ ]:
# @title Code: fig-parallel-coordinates
#| label: fig-parallel-coordinates
#| eval: false
#| fig-cap: "Interactive parallel coordinates plot. Each line represents a Pareto-front solution. Drag axis ranges to filter solutions interactively. Colour indicates O1 (total agricultural deficit)."
decision_vars = list(result.solutions[0].parameters)[:8]
fig = plot_parallel_coordinates(result, OBJ_NAMES, decision_vars=decision_vars)
figures.caption(
    fig,
    "Interactive parallel coordinates plot. Each line represents a Pareto-front solution and colour indicates O1 total agricultural deficit.",
    label="parallel-coordinates",
)

## Representative Solutions {#sec-representatives}

The Pareto front contains many solutions, but for practical decision-making it helps to extract a small number of **representative solutions** that illustrate the range of achievable tradeoffs. We select five: one that minimises each of the four objectives individually (the "extreme" solutions), plus a **balanced** compromise that sits closest to the ideal point (the hypothetical solution that achieves the best value on every objective simultaneously).

@tbl-representatives shows the objective values for these five representatives. The extremes reveal the "price" of specialising on one objective: the solution that minimises agricultural deficit (Min Ag. Deficit) achieves the lowest O1 but may sacrifice equity (O6) or smoothness (O9). Conversely, the Min Equity solution achieves the fairest distribution but at the cost of higher total deficit. The Balanced solution represents a middle ground --- not the best on any single objective, but reasonable across all four.

In [ ]:
# @title Code: tbl-representatives
#| label: tbl-representatives
#| eval: false
#| tbl-cap: "Representative solutions from the Pareto front, selected by minimising each objective individually plus a balanced compromise."
representatives = representative_indices(front_obj, OBJ_NAMES)
tables.caption(
    representative_table(front_obj, representatives, OBJ_NAMES).style.hide(axis="index"),
    "Representative solutions from the Pareto front, selected by minimising each objective individually plus a balanced compromise.",
    label="representatives",
)

@fig-district-breakdown breaks down O1 (total agricultural deficit) into per-district contributions for each representative solution. This reveals which districts bear the cost of different tradeoff strategies --- the aggregate objective values from the table above hide the geographic distribution of benefits and losses.

In [ ]:
# @title Code: fig-district-breakdown
#| label: fig-district-breakdown
#| eval: false
#| fig-cap: "Per-district deficit breakdown for representative solutions. Different solutions sacrifice different districts, revealing the geographic dimension of the tradeoff."
rep_systems = simulate_representatives(result, representatives, TIMESTEPS)
fig = plot_representative_district_deficits(rep_systems)
figures.caption(
    fig,
    "Per-district deficit breakdown for representative solutions, revealing the geographic distribution of each tradeoff strategy.",
    label="district-breakdown",
)

Beyond the irrigation districts, the representative solutions have implications for **non-objective flows** that stakeholders care about: interbasin transfers to adjacent basins (sink flows) and cooling water availability at the power plant. @fig-sink-powerplant shows the total flow received by each sink node and the Powerplant across the five representative solutions. These flows are not directly optimised but emerge as consequences of the allocation decisions.

In [ ]:
# @title Code: fig-sink-powerplant
#| label: fig-sink-powerplant
#| eval: false
#| fig-cap: "Total flow to sink nodes and the Powerplant under representative solutions. These flows are not directly optimised but are consequences of the allocation decisions. Differences across solutions reveal how upstream tradeoffs propagate to interbasin commitments and power generation reliability."
fig = plot_representative_sink_flows(rep_systems, TIMESTEPS)
figures.caption(
    fig,
    "Total flow to sink nodes and the Powerplant under representative solutions, showing downstream consequences of upstream tradeoffs.",
    label="sink-powerplant",
)

The sink flows represent interbasin water transfers and regulatory minimum flow commitments --- Sink_Jizzakh, Sink_Kashkadarya, and Sink_Navoi deliver water to adjacent basins that depend on the Zarafshan for part of their supply. The Powerplant flow represents cooling water availability for power generation. Differences across the representative solutions show how upstream allocation tradeoffs (the objectives we optimise) propagate to downstream commitments (that we monitor but do not directly optimise).

## What Did the Optimizer Learn? {#sec-optimizer-learned}

@fig-parameters provides a direct view into what the optimizer "learned" about the system's physical structure. Each position on the x-axis represents a selected policy variable. For each representative solution, the variable's value is plotted as a dot, normalised to the [0, 1] range (where 0 = lower bound and 1 = upper bound) so that variables are visually comparable regardless of their physical units.

The key insight comes from comparing the vertical spread of dots at each variable position. Variables where all five dots cluster at nearly the same height are **consensus parameters**: the optimizer arrives at essentially the same value regardless of which objective is prioritised. These represent settings where the physics of the system leaves little room for manoeuvre --- the optimal value is determined by the network topology, infrastructure capacity, or demand distribution, not by the decision-maker's preference among objectives. Consensus parameters can be implemented with confidence, as they represent robust operational choices.

In contrast, variables where the dots spread widely across the [0, 1] range are the **true operational levers**: different tradeoff strategies require fundamentally different settings. These are the parameters where stakeholder preferences actually matter --- a decision-maker who prioritises equity will set them differently from one who prioritises total deficit minimisation. Understanding which parameters are consensus and which are levers is arguably the most actionable insight from the entire optimisation: it tells the water manager where they have genuine flexibility and where the system's physics has already decided for them.

In [ ]:
# @title Code: fig-parameters
#| label: fig-parameters
#| eval: false
#| fig-cap: "Decision variable values for representative solutions, normalised to [0, 1] bounds. Parameters where all dots cluster (tight vertical spread) are 'consensus' parameters locked in by the physics. Parameters where dots spread widely are the true operational levers where stakeholder preferences determine the setting."
fig = plot_selected_parameter_profiles(result, representatives, system.param_bounds())
figures.caption(
    fig,
    "Decision variable values for representative solutions, normalised to [0, 1] bounds. Tight clusters indicate consensus parameters; wide spreads indicate operational levers.",
    label="parameters",
)

Consider these guided questions when interpreting the results:

-   **Which e-flow fraction deviates most from what a human operator might set?** What physical condition (e.g., demand patterns, downstream topology) does this compensate for?
-   **Does the optimizer favour aggressive or conservative Kattakurgan release?** How does `buffer_coef` differ across representative solutions --- and what does this say about the value of drought insurance?
-   **Do equity-optimal and deficit-optimal solutions use Kattakurgan differently?** What does this tell you about the role of storage in fairness?
-   **Compare the smoothest solution's reservoir release pattern to the min-deficit solution's.** What operational reality does the smoothness objective enforce?
-   **Did the optimizer discover anything non-obvious** --- a parameter setting that a human operator might not have considered?

::: {.callout-tip title="Exercise 6: Reading the Pareto Front" collapse="true"}
**Task:** Using the pairwise scatter plots from @fig-pareto:

1.  Identify the pair of objectives with the steepest tradeoff (where small improvements in one cause large degradations in the other). What physical mechanism drives this conflict?
2.  Identify any pair of objectives that appear positively correlated (improving one also improves the other). Does this suggest one of them is redundant?
3.  How many non-dominated solutions have O4 = 0 (zero power plant deficit)? What is the range of O1 values among these solutions?
4.  Pick a point on the boundary of the O1 vs. O4 panel. Locate the same solution in the O6 vs. O9 panel (hint: use the parallel coordinates plot to identify it). Is it still on the boundary, or has it moved into the interior? What does this tell you about the solution's trade-off strategy?
5.  If we had only 2 objectives (say O1 and O4), would it be correct to connect the dots into a continuous Pareto front line? Refer to @sec-reading-scatter-matrix for why the answer differs between 2 and 4 objectives.
:::

::: {.callout-tip title="Exercise 7: Navigating the Parallel Coordinates" collapse="true"}
**Task:** Using the interactive parallel coordinates plot from @fig-parallel-coordinates:

1.  Drag the O4 axis to select only solutions with zero power plant deficit. How many solutions remain? What is the range of O1 (total deficit) in this filtered set?
2.  Within this filtered set, further restrict to solutions where O6 (equity) is below the median value. What is the cost in O9 (smoothness) --- i.e., what range of O9 do these solutions span?
3.  Examine the decision variable axes for this doubly-filtered set. Which parameter shows the least variation (is most "locked in" by the constraints)? Which shows the most variation (offering the most operational flexibility)?
4.  Summarise: what is the "price" of combining power plant reliability with equity, measured in smoothness and total deficit?
:::

::: {.callout-important title="Key Takeaways -- Optimisation Results"}
-   The Pareto front reveals the full landscape of achievable tradeoffs among the four objectives.
-   Representative solutions illuminate *how* different parameter settings lead to different tradeoff positions.
-   The interactive parallel coordinates plot is the primary tool for navigating the front.
-   Understanding *what the optimizer learned* (parameter interpretation) is as important as the objective scores it achieved.
:::

# Discussion {#sec-discussion}

## What Optimisation Reveals {#sec-what-reveals}

The Pareto front quantifies fundamental tradeoffs in the Zarafshan basin. Connect this back to @sec-building-intuition: the tradeoffs students discovered manually are now mapped exhaustively. The optimizer didn't invent new physics --- it systematically explored consequences the students already glimpsed.

Key structural insights:

-   **Irrigation vs. equity**: Total agricultural deficit (O1) and maximum district deficit (O6) conflict because the sequential network topology creates systematic disadvantage for downstream districts. Reducing total deficit concentrates water upstream where demands are first served.
-   **Reliability vs. performance**: Maintaining zero power plant deficit (O4) constrains the allocation space, potentially increasing total agricultural deficit. HW_Karmana's local split policy and downstream canal losses determine how much of the routed water can actually serve the Powerplant.
-   **Smoothness vs. optimality**: The release variability objective (O9) penalises solutions that achieve low deficits through erratic reservoir operations. This is operationally necessary but comes at a cost.

## Choosing a Solution: The Water Minister's Dilemma {#sec-ministers-dilemma}

> *The Water Resources Ministry must select one operating policy from the Pareto front. Four stakeholders sit at the table:*
>
> -   *The **Irrigation Department** wants to minimise total agricultural deficit (O1).*
> -   *The **Equity Commission** insists no single district bears more than 35% of total deficit (O6 constraint).*
> -   *The **Power Plant operator** requires zero cooling deficit (O4 = 0).*
> -   *The **Operations Division** says day-to-day release changes must stay below a threshold they specify (O9 constraint).*
>
> *Using the parallel coordinates plot from @fig-parallel-coordinates, identify the feasible solution set that satisfies all four stakeholders simultaneously. How many solutions remain? What is the achievable range of O1? What happens if the Equity Commission relaxes their constraint to 40% --- how many additional solutions become available, and how much does the best O1 improve?*
>
> *This scenario illustrates the fundamental insight of multi-objective optimisation: the Pareto front provides the menu; stakeholders must negotiate which dish to order.*

## What Optimisation Cannot Tell You {#sec-limitations-intro}

The Pareto front answers "what is achievable?" but not "what should we choose?" Selection requires human values, negotiation, and political judgement. The optimizer is a decision-support tool, not a decision-making tool.

## Model Limitations {#sec-model-limitations}

Several simplifications should be kept in mind:

-   **Perfect foresight**: The optimizer knows the entire 2191-day inflow sequence in advance. Real operators face forecast uncertainty.
-   **Single historical sequence**: Results are conditioned on 2017--2022 hydrology. A different climate sequence would produce a different Pareto front.
-   **Daily timestep**: Sub-daily dynamics (flood routing, within-day demand peaks) are averaged out.
-   **Continuous variables**: Real operations involve discrete decisions (gate positions, scheduled releases).
-   **No operating costs**: The model treats all water movement as cost-free.

## Future Directions {#sec-future-directions}

-   **Alternative seasonal parameterisations**: Reducing reservoir SLOP values from 12 monthly values to 4 seasonal blocks could improve interpretability and lower the search dimension; expanding e-flow fractions seasonally would add flexibility but also increase dimensionality.
-   **Climate scenarios**: Running the optimizer under synthetic drought scenarios would test robustness.
-   **Robust optimisation**: Optimising over multiple inflow sequences simultaneously would identify solutions that perform well under uncertainty.
-   **Seasonal grouping**: Defining 4 seasonal SLOP blocks instead of 12 monthly values could balance flexibility with tractability.

::: {.callout-tip title="Exercise H: Stakeholder Perspectives" collapse="true"}
**Task:** You are advising the Uzbek Ministry of Water Resources on the Zarafshan allocation. A drought year is expected (20% reduction in source inflow).

1.  Which of the four objectives (O1, O4, O6, O9) would you prioritise, and why? Would your answer change if the drought were expected to last 3 years?
2.  The current optimisation uses scalar e-flow fractions and monthly-cyclical reservoir policies. What advantage would seasonally varying e-flow fractions offer in a drought scenario? What disadvantage?
3.  A downstream province argues that the optimisation unfairly privileges upstream districts because it doesn't include O5 (inter-basin transfer deficit). How would you respond?
:::

::: {.callout-important title="Key Takeaways -- Discussion"}
-   The Pareto front quantifies tradeoffs; human values select the solution.
-   Optimisation is conditioned on model assumptions and a single historical sequence.
-   Future work could expand the decision space, incorporate climate scenarios, and test robustness.
:::

# Conclusions {#sec-conclusions}

## Summary of Findings {#sec-summary}

This coursebook demonstrated multi-objective optimisation of the Zarafshan River Basin water allocation system:

1.  **Policy parameters by design**: EFlowSplitPolicy collapses splitter ratios into single e-flow fractions, while reservoir SLOP policies expose seasonal operating values instead of daily controls.
2.  **Four objectives capture the key tradeoffs**: Agricultural deficit (O1), power plant reliability (O4), equity (O6), and operational smoothness (O9) span the primary dimensions of system performance.
3.  **The Pareto front reveals structural tradeoffs**: Equity and total deficit conflict because of the sequential network topology. Smoothness and performance conflict because optimal reservoir operations require erratic releases. These are physical realities, not artifacts of the model.
4.  **The optimizer's parameter choices are interpretable**: By examining *how* different Pareto solutions set policy parameters, we can understand what the optimizer "learned" about the system and what operational changes would be most impactful.

## Practical Implications {#sec-practical-implications}

For a water manager:

-   The Pareto front is a menu, not a prescription. Use it to understand what is achievable and at what cost.
-   The interactive parallel coordinates plot is the primary decision-support tool. Filter by hard constraints first (O4 = 0, O9 below threshold), then explore the remaining options.
-   Parameters that show little variation across the front are "locked in" by the physics --- there is little to gain by adjusting them. Parameters that vary widely are the true operational levers.
-   The priority rule at HW_Dargom demonstrates that some decisions are better handled by deterministic rules than by optimisation. EFlowSplitPolicy at corridor splitters demonstrates how policy design can encode physical knowledge to reduce dimensionality.

# Exercise Collection {#sec-exercises}

## Conceptual / Discussion Exercises (Letters)

-   **Exercise D --- Counting Decision Variables** (@sec-reducing-complexity): Analyse the dimensionality of the current monthly-cyclical SLOP schema and possible seasonal reductions.
-   **Exercise E --- Priority Rules and Optimizer Signals** (@sec-reducing-complexity): Explain how the HW_Dargom priority rule and EFlowSplitPolicy reduce the optimisation space.
-   **Exercise F --- Adding a Fifth Objective** (@sec-four-objectives): Predict the effect of adding O2 on front dimensionality and correlation structure.
-   **Exercise G --- Effect of Freeing More Knobs** (@sec-opt-question): Analyse the consequences of replacing monthly SLOP values with coarser seasonal groups.
-   **Exercise H --- Stakeholder Perspectives** (@sec-future-directions): Policy discussion on objective prioritisation under drought.

## Computational Exercises (Numbers)

5.  **Exercise 5 --- Manual Tradeoff Discovery** (@sec-impossible-tradeoff): Manually vary two parameters, plot attempts in objective space, compare to the Pareto front.
6.  **Exercise 6 --- Reading the Pareto Front** (@sec-pareto-front): Identify tradeoff slopes, correlations, and feasible regions from pairwise scatter plots. Understand why 2D projections of a 4D Pareto front cannot be connected into lines (@sec-reading-scatter-matrix).
7.  **Exercise 7 --- Navigating the Parallel Coordinates** (@sec-parallel-coordinates): Guided interactive exploration --- filter by constraints, examine parameter variation, quantify the price of combining objectives.

The exercises progress from building physical intuition (5), through formal understanding of decision and objective spaces (D, E, F, G), to quantitative analysis of optimisation results (6, 7) and policy discussion (H).

# Appendix A: Configuration Details {#sec-appendix-config}

## Full Parameter Inventory

The complete parameter inventory is documented in @tbl-full-knob-inventory. It is discovered from `WaterSystem.param_schema()` so the coursebook reflects the actual code path.

## Parameter Schema

@tbl-reduced-config defines the decision variables with per-family bounds. The bounds are physically motivated:

-   **E-flow fractions**: \[0, 1\], spanning fully irrigation-only (0) to fully environmental (1).
-   **SLOP `vr`**: \[0, 500\] m³/s, the declared bound on release rate.
-   **`v1` and `v2`**: \[0, 10⁹\] m³. Constraint: v1 ≤ v2.
-   **`buffer_coef`**: \[0, 1\], full range of reduction factors.
-   **`flood_coef`**: \[0, 5\], flood-zone release multiplier.

## Constraint Specifications

**Ordered (v1 ≤ v2)**: Both reservoirs have an `Ordered(low="v1", high="v2")` constraint. taqsim's `make_repair()` factory generates a repair function that swaps v1 and v2 if violated after any genetic operation.

No sum-to-one constraints are needed: splitter ratios are no longer decision variables (they are either single e-flow fractions or demand-weighted remainders).

## Optimizer Hyperparameters

| Parameter        | Value | Justification                                       |
|:-----------------------|:-----------------------|:-----------------------|
| `pop_size`       | 12 smoke / 300 demo | Smoke default prevents accidental long test runs; demo mode gives a modest front |
| `n_generations`  | 2 smoke / 100 demo | Smoke default checks the pipeline only; convergence diagnostics determine whether to extend demo runs |
| `eta` (SBX)      | 15.0  | Standard value for moderate variable interaction    |
| `eta` (mutation) | 20.0  | Standard value for local search refinement          |
| `n_workers`      | 2 smoke / 6 demo | Smoke mode stays light; demo mode uses moderate parallelism |
| `seed`           | 42    | Reproducibility                                     |

## Repair Function Logic

```         
taqsim.optimization.make_repair() auto-generates:
1. clip(x, lower, upper)            # Box constraints
2. For each Ordered constraint:
     if v1 > v2: swap(v1, v2)       # Ordered(v1 ≤ v2)
```

## Operator Composition

Handled internally by `taqsim.optimization.optimize()` via `run_optimization()`. The optimizer composes SBX crossover, polynomial mutation, and the auto-generated repair function.

# Appendix B: Glossary {#sec-glossary}

| Term | Definition |
|:-----------------------|:----------------------------------------------|
| **Bounds** | Per-variable minimum and maximum values that define the feasible region of the decision space. |
| **Convergence** | The stabilisation of the Pareto front over successive generations, measured by hypervolume. |
| **Crowding distance** | A density metric: solutions in less-crowded regions of the front are preferred to maintain diversity. |
| **Decision variable** | A parameter value that the optimizer can adjust. In this coursebook these include splitter e-flow fractions and monthly-cyclical reservoir SLOP values. |
| **Dominance** | Solution A *dominates* B if A is no worse on all objectives and strictly better on at least one. |
| **Hypervolume** | The volume of objective space dominated by the Pareto front relative to a reference point. Larger is better. |
| **Knee point** | A solution where the tradeoff slope changes most sharply --- maximum improvement per unit of degradation. |
| **Non-dominated** | A solution that is not dominated by any other solution in the population. |
| **Objective function** | A function mapping a parameter configuration to a scalar performance measure (e.g., total agricultural deficit). |
| **Pareto front** | The set of all non-dominated solutions. Improving one objective necessarily worsens another. |
| **Pareto rank** | Rank 0 = the Pareto front; rank 1 = the front after removing rank 0; and so on. |
| **Repair function** | A function that projects infeasible solutions back into the feasible region after genetic operations. |

# References {.unnumbered}